# SHAP → COGNITION ANALYSIS PIPELINE

In [69]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, LinearRegression

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests


# 1. LOAD DATA

In [70]:
shap_h = pd.read_csv("shap_matrix_h_QSM_mean.csv", header=0) # (59, features)
feature_names = shap_h.columns[1:].tolist()  # Assuming first column is subject_id and rest are features
shap_h.columns = ["subject_id"] + feature_names
shap_MA = pd.read_csv("shap_matrix_MA_QSM_mean.csv", header=0) # (5, features)
shap_MA.columns = ["subject_id"] + feature_names
# concatenate shap_h and shap_MA
shap = pd.concat([shap_h, shap_MA], axis=0).reset_index(drop=True)

df = pd.read_csv("biological_metrics_MCIandAD/merged_results_with_metadata.csv", header=0) # (N, metrics)

# align df by subject_id in shap and cut df to only those subjects
df = df[df["subject_id"].isin(shap["subject_id"])]
# sort df by subject_id
df = df.sort_values("subject_id").reset_index(drop=True)
# sort shap by subject_id
shap = shap.sort_values("subject_id").reset_index(drop=True)


# 2. DEFINE VARIABLES

In [79]:

cognition_vars = [
    "Memory_Composite",
    "Executive_Function_Composite",
    "Processing_Speed_Composite",
    "Language_Composite",
    "Visuospatial_Composite",
    "Global_Cognition_Composite"
]

covariates = ["age", "sex_numeric", "APOE4"]

save_dir = "shap_cognition_QSM_results"


# 3. OPTIONAL: AGGREGATE LAMINAR → REGION

In [72]:
# If features = region_layer, e.g. "precuneus_L3"
if any("_L" in f for f in feature_names):
    region_names = sorted(list(set([f.split("_L")[0] for f in feature_names])))

    shap_region = np.zeros((shap.shape[0], len(region_names)))

    for i, region in enumerate(region_names):
        idx = [j for j, f in enumerate(feature_names) if f.startswith(region)]
        shap_region[:, i] = shap[:, idx].mean(axis=1)

else:
    shap_region = shap
    region_names = feature_names

shap_df = pd.DataFrame(shap_region, columns=region_names)


# 4. STANDARDIZE SHAP

In [73]:

scaler = StandardScaler()
shap_scaled = scaler.fit_transform(shap_df)


# 5. LASSO FEATURE SELECTION (PER DOMAIN)

In [74]:
# ignore warning for LassoCV convergence
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.linear_model._coordinate_descent")

lasso_results = {}

for cog in cognition_vars:
    mask = df[cog].notna()

    X = shap_scaled[mask]
    y = df.loc[mask, cog].values

    lasso = LassoCV(cv=5, max_iter=5000).fit(X, y)

    coefs = pd.Series(lasso.coef_, index=region_names)
    coefs = coefs[coefs != 0].sort_values()

    lasso_results[cog] = coefs

    print(f"\n=== {cog} ===")
    print(coefs)



=== Memory_Composite ===
rh_precentral    -0.092932
rh_paracentral   -0.034607
dtype: float64

=== Executive_Function_Composite ===
rh_caudalmiddlefrontal   -8.386742e-17
dtype: float64

=== Processing_Speed_Composite ===
lh_caudalanteriorcingulate   -0.021764
dtype: float64

=== Language_Composite ===
lh_isthmuscingulate    5.553335e-17
dtype: float64

=== Visuospatial_Composite ===
lh_cuneus   -1.114894e-16
dtype: float64

=== Global_Cognition_Composite ===
rh_superiorparietal   -0.015231
dtype: float64


# 6. BUILD SHAP COGNITIVE SCORE

In [75]:

shap_scores = {}

for cog in cognition_vars:
    coefs = lasso_results[cog]
 
    if len(coefs) == 0:
        continue

    X_sub = shap_df[coefs.index].values
    score = X_sub @ coefs.values

    shap_scores[cog] = score
    df[f"{cog}_SHAPscore"] = score


# 7. GLM WITH COVARIATES (INFERENCE)

In [76]:

glm_results = []

for cog in cognition_vars:
    score_col = f"{cog}_SHAPscore"

    if score_col not in df.columns:
        continue

    data = df[[cog, score_col] + covariates].dropna()

    y = data[cog]
    X = data[[score_col] + covariates]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    glm_results.append({
        "cognitive_domain": cog,
        "beta_SHAP": model.params[score_col],
        "p_SHAP": model.pvalues[score_col],
        "R2": model.rsquared
    })

glm_df = pd.DataFrame(glm_results)

# FDR correction
glm_df["p_FDR"] = multipletests(glm_df["p_SHAP"], method="fdr_bh")[1]

print("\n=== GLM RESULTS ===")
print(glm_df)



=== GLM RESULTS ===
               cognitive_domain     beta_SHAP    p_SHAP        R2     p_FDR
0              Memory_Composite  2.413162e+01  0.006477  0.429228  0.019430
1  Executive_Function_Composite  4.522256e-15  0.248640  0.035287  0.298368
2    Processing_Speed_Composite  2.037541e+02  0.045514  0.404919  0.091027
3            Language_Composite -2.167222e-15  0.650823  0.003700  0.650823
4        Visuospatial_Composite  7.684674e-15  0.001536  0.256989  0.009219
5    Global_Cognition_Composite  1.144954e+02  0.111769  0.294203  0.167653


# 8. REGION-WISE INFERENCE (OPTIONAL, STRONG FIGURE)

In [77]:

region_stats = []

for cog in cognition_vars:
    for region in region_names:

        data = df[[cog] + covariates].copy()
        data["region_SHAP"] = shap_df[region]

        data = data.dropna()

        y = data[cog]
        X = data[["region_SHAP"] + covariates]
        X = sm.add_constant(X)

        model = sm.OLS(y, X).fit()

        region_stats.append({
            "cognitive_domain": cog,
            "region": region,
            "beta": model.params["region_SHAP"],
            "p": model.pvalues["region_SHAP"]
        })

region_df = pd.DataFrame(region_stats)

# FDR per domain
region_df["p_FDR"] = region_df.groupby("cognitive_domain")["p"].transform(
    lambda p: multipletests(p, method="fdr_bh")[1]
)


# 9. SAVE OUTPUTS

In [80]:
from os import mkdir

mkdir(save_dir)
glm_df.to_csv(f"{save_dir}/SHAP_cognition_GLM.csv", index=False)
region_df.to_csv(f"{save_dir}/SHAP_region_cognition.csv", index=False)

# Save top regions
top_regions = region_df[region_df["p_FDR"] < 0.05]
top_regions.to_csv(f"{save_dir}/SHAP_significant_regions.csv", index=False)

print("\nSaved all outputs.")


Saved all outputs.
